# Vision Transformer (ViT) from Scratch — CIFAR-10

A Vision Transformer implemented **from scratch in PyTorch** (no `timm`, no pretrained
models) and trained on CIFAR-10.

**Architecture built from first principles:**
- Patch embedding via strided convolution
- Learnable `[CLS]` token + positional embeddings
- Multi-head self-attention
- Transformer encoder blocks (pre-norm, residual connections, GELU MLP)
- Full ViT assembly + training loop

**Result:** 683K-parameter model → **75% test accuracy** on CIFAR-10 (20 epochs, AdamW + augmentation).


## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Patch Embedding

Splits a 32×32 image into 4×4 patches (64 patches) using a strided conv that also
projects each patch to a 96-dim embedding. Prepends a learnable `[CLS]` token and adds
learnable positional embeddings, giving a sequence of 65 tokens.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=96):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        # Strided conv: cuts image into non-overlapping patches AND projects them
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))

    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x).flatten(2).transpose(1, 2)   # (B, 64, 96)
        cls = self.cls_token.expand(B, -1, -1)         # (B, 1, 96)
        x = torch.cat([cls, x], dim=1)                 # (B, 65, 96)
        x = x + self.pos_embed                         # add positional info
        return x


# Sanity check
pe = PatchEmbedding()
print(pe(torch.randn(2, 3, 32, 32)).shape)   # expect (2, 65, 96)

## 3. Multi-Head Self-Attention

Scaled dot-product attention split across 4 heads. Q, K, V are produced by a single
fused linear layer, reshaped into heads, attention is computed per head, then heads are
merged and projected back to the embedding dimension.

`Attention(Q, K, V) = softmax(QKᵀ / √d_k) · V`

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim=96, num_heads=4):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads          # 96 / 4 = 24
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)  # Q, K, V in one layer
        self.proj = nn.Linear(embed_dim, embed_dim)     # final projection

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)                # (3, B, heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B, heads, N, N)
        weights = F.softmax(scores, dim=-1)
        out = weights @ v                               # (B, heads, N, head_dim)

        out = out.transpose(1, 2).reshape(B, N, C)      # merge heads -> (B, N, 96)
        return self.proj(out)


# Sanity check
mha = MultiHeadAttention()
print(mha(torch.randn(2, 65, 96)).shape)   # expect (2, 65, 96)

## 4. Transformer Encoder Block

Pre-norm transformer block: LayerNorm → attention → residual, then
LayerNorm → MLP → residual. Shape is preserved so blocks can be stacked.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=96, num_heads=4, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden = embed_dim * mlp_ratio                  # 96 * 4 = 384
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, embed_dim),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))                # attention + residual
        x = x + self.mlp(self.norm2(x))                 # MLP + residual
        return x


# Sanity check
block = TransformerBlock()
print(block(torch.randn(2, 65, 96)).shape)   # expect (2, 65, 96)

## 5. Full Vision Transformer

Assembles the pieces: patch embedding → stack of 6 transformer blocks → LayerNorm →
read the `[CLS]` token → linear classification head producing 10 class logits.

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 embed_dim=96, num_heads=4, depth=6, num_classes=10):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        cls = x[:, 0]                                   # take CLS token
        return self.head(cls)


model = VisionTransformer()
print("Output shape:", model(torch.randn(2, 3, 32, 32)).shape)   # expect (2, 10)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters")

## 6. Data — CIFAR-10

Training set uses augmentation (random crop + horizontal flip); test set is kept clean.
Both are normalized with CIFAR-10 channel statistics.

> **Note:** set `DATA_ROOT` to the folder containing `cifar-10-batches-py`.
> On Kaggle, attach the *CIFAR-10 Python* dataset and use its input path;
> locally, set `download=True`.

In [ ]:
# Kaggle: point to the attached CIFAR-10 dataset. Locally: use './data' with download=True.
DATA_ROOT = '/kaggle/input/datasets/pankrzysiu/cifar10-python'
DOWNLOAD = False

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
test_tf = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_ds = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True,  download=DOWNLOAD, transform=train_tf)
test_ds  = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=DOWNLOAD, transform=test_tf)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

## 7. Training Setup

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = VisionTransformer().to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

# Sanity check: untrained loss should be ~ln(10) = 2.30
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)
print(f"Device: {device}")
print(f"Initial loss: {criterion(model(images), labels).item():.4f}  (expect ~2.30)")

## 8. Training Loop

Standard train/eval loops. Training accumulates loss and accuracy per epoch; evaluation
runs under `torch.no_grad()` in `eval()` mode.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


EPOCHS = 20
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["test_loss"].append(te_loss);  history["test_acc"].append(te_acc)
    print(f"Epoch {epoch:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"test loss {te_loss:.3f} acc {te_acc:.3f}")

print(f"\nBest test accuracy: {max(history['test_acc']):.3f}")

## 9. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, history["train_loss"], label="train")
ax1.plot(epochs_range, history["test_loss"], label="test")
ax1.set_title("Loss"); ax1.set_xlabel("epoch"); ax1.legend()

ax2.plot(epochs_range, history["train_acc"], label="train")
ax2.plot(epochs_range, history["test_acc"], label="test")
ax2.set_title("Accuracy"); ax2.set_xlabel("epoch"); ax2.legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
plt.show()

## 10. Save Model

In [ ]:
torch.save(model.state_dict(), "vit_cifar10.pth")
print("Model saved to vit_cifar10.pth")